In [ ]:
import pandas as pd
import feature

In [27]:
df=pd.read_csv("../data/Amazon_Sale_Report.csv")


C:\Users\ABHIJITH\AppData\Local\Temp\ipykernel_10404\2237359375.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("../data/Amazon_Sale_Report.csv")


In [28]:
df = df.drop(columns=['Unnamed: 22', 'fulfilled-by'])


In [29]:
df['promotion']=df['promotion'].fillna('No promotion')

In [30]:
multiple = ['ship-city', 'ship-state', 'ship-postal-code','currency']
df[multiple] = df[multiple].fillna('Unknown')

In [31]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')

In [32]:
df['ship-country']=df['ship-country'].fillna('IN')

In [33]:
df.loc[df['status'] == 'Cancelled', 'courier_status'] = 'Cancelled'
df.loc[df['status'] == 'Shipped - Delivered to Buyer', 'courier_status'] = 'Shipped'
df.loc[df['status'] == 'Shipped - Returned to Seller', 'courier_status'] = 'Shipped'

In [34]:
df.amount=df.amount.fillna(0)

In [35]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

C:\Users\ABHIJITH\AppData\Local\Temp\ipykernel_10404\2370506791.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [36]:
df['date'] = df['date'].ffill()

In [37]:
df['month']=df['date'].dt.strftime('%B')

In [38]:
df['year']=df.date.dt.year

In [39]:
df['day_name']=df.date.dt.strftime('%A')

In [40]:
df['is_weekend']=df.date.dt.dayofweek.isin([5,6]).astype(int)

In [41]:
df['date'] = df['date'].dt.strftime('%Y-%m-%d')

In [43]:
sku_sales_velocity=df.groupby('sku')['qty'].transform('sum')


In [44]:
df['item_velocity']=pd.qcut(sku_sales_velocity,q=3,labels=['Slow Moving', 'Moderate', 'Fast Selling'])

In [45]:
category_counts = df.groupby('category')['qty'].sum().reset_index()
category_counts = category_counts.sort_values(by='qty', ascending=False).reset_index(drop=True)

category_counts['category_demand']=category_counts.index.map(feature.index_var)
category_dict=dict(zip(category_counts['category'],category_counts['category_demand']))


In [46]:
df['category_demand']=df.category.map(category_dict)

In [47]:
df.columns

Index(['index', 'order_id', 'date', 'status', 'fulfilment', 'sales_channel_',
       'ship-service-level', 'style', 'sku', 'category', 'size', 'asin',
       'courier_status', 'qty', 'currency', 'amount', 'ship-city',
       'ship-state', 'ship-postal-code', 'ship-country', 'promotion', 'b2b',
       'month', 'year', 'day_name', 'is_weekend', 'item_velocity',
       'category_demand'],
      dtype='object')

In [48]:
%pip install sqlalchemy pymysql


Note: you may need to restart the kernel to use updated packages.


In [50]:

from sqlalchemy import create_engine
DB_USER = "root"       
DB_PASSWORD = "root"   
DB_HOST = "localhost"          
DB_PORT = "3306"                
DB_NAME = "Amazon_sales_data"  


connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)


In [51]:
df.to_sql("amazon_sales_report_cleaned", con=engine, if_exists="replace", index=False)

128975